# Rhythmicity of the gut microbiota and sleep in infants 

Summary: Analysis of the rhythmicity in the samples (gut microbiota rhythmicity and sleep rhythmicity).

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from functions_script import load_tsv, scatterplot, plot_estimates, plot_estimates_vertically

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

In [ ]:
# create color palette
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]

## Load infants metadata

In [ ]:
# load infant metadata per age/timepoint
md_ages = load_tsv(f"{meta_data_path}/md_ages.tsv")
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

## Alpha diversity rhythmicity (and sleep rhythmicity) along age

### Alpha diversity rhythmicity

In [ ]:
# filter data for microbiota rhytmicity values
R2_cols = ['R2_abundance', 'R2_abundance_evenness', 'R2_evenness', 'R2_biodiversity']
data_R2 = md_ages.dropna(subset=R2_cols)

In [ ]:
# linear mixed effects model for observed features rhythmicity
model_of = smf.mixedlm("R2_abundance ~ age_days + sex",
                    data_R2, 
                    groups=data_R2["infant_id"]).fit()
print(model_of.summary())

In [ ]:
# linear mixed effects model for Shannon entropy rhythmicity
model_sh = smf.mixedlm("R2_abundance_evenness ~ age_days + sex",
                    data_R2, 
                    groups=data_R2["infant_id"]).fit()
print(model_sh.summary())

In [ ]:
# linear mixed effects model for Pielou evenness rhythmicity
model_pi = smf.mixedlm("R2_evenness ~ age_days + sex",
                    data_R2, 
                    groups=data_R2["infant_id"]).fit()
print(model_pi.summary())

In [ ]:
# linear mixed effects model for Faith phylogenetic diversity rhythmicity
model_fa = smf.mixedlm("R2_biodiversity ~ age_days + sex",
                    data_R2, 
                    groups=data_R2["infant_id"]).fit()
print(model_fa.summary())

In [ ]:
# scatter plots of alpha diversity rhythmicity scores over age
fig, axs = plt.subplots(1, 4, figsize=(16, 6))

scatterplot(data_R2, "age_days", "R2_abundance", "timepoint", 
                    "", "Alpha diversity rhythmicity: \n cosine fit score of observed features", "Age (days)",
                    timepoints_colors, axs[0], loc_position = 'upper center')

scatterplot(data_R2, "age_days", "R2_abundance_evenness", "timepoint", 
                    "", "Alpha diversity rhythmicity: \n cosine fit score of Shannon entropy", "Age (days)",
                    timepoints_colors, axs[1], loc_position=None)

scatterplot(data_R2, "age_days", "R2_evenness", "timepoint", 
                    "", "Alpha diversity rhythmicity: \n cosine fit score of Pielou evenness", "Age (days)",
                    timepoints_colors, axs[2], loc_position=None)

scatterplot(data_R2, "age_days", "R2_biodiversity", "timepoint", 
                    "", "Alpha diversity rhythmicity: \n cosine fit score of Faith's phylogenetic diversity", "Age (days)",
                    timepoints_colors, axs[3], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/alpha_diversity_rhythmicity.pdf", dpi=300, bbox_inches='tight')
plt.show()

### Sleep rhythmicity

In [ ]:
# filter data for sleep model
data_CFI = md_ages.dropna(subset=["CFI"])
print(data_CFI.shape)

In [ ]:
# check number of assessment days
data_CFI["CFI_assessment_days"].describe()

In [ ]:
# linear mixed effects model for sleep rhythmicity
model_sleep = smf.mixedlm("CFI ~ age_days + sex",
                    data_CFI, 
                    groups=data_CFI["infant_id"]).fit()
print(model_sleep.summary())

### Alpha diversity rhythmicity and sleep rhythmicity along age

In [ ]:
# coefficient plots for alpha diversity rhythmicity and sleep rhythmicity models
models = [model_of, model_sh, model_pi, model_fa, model_sleep]
titles = ["Alpha diversity rhythmicity: \n cosine fit score of \n observed features", 
          "Alpha diversity rhythmicity: \n cosine fit score of \n Shannon entropy", 
          "Alpha diversity rhythmicity: \n cosine fit score of \n  Pielou evenness", 
          "Alpha diversity rhythmicity: \n cosine fit score of \n Faith's phylogenetic diversity", 
          "Sleep rhythmicity: \n Circadian function index \n (CFI)"]
x_pos = [0.02, 0.2, 0.05, 0.00, 0.06]
plot_estimates(models, titles, x_pos, f"{figures_path}/rhythmicity_estimates.pdf")

## Alpha diversity rhythmicity and sleep rhythmicity

In [ ]:
# filter data for the combined analysis
data_rhythmicity = md_ages.dropna(subset=R2_cols + ["CFI"]).copy()
print(data_rhythmicity.shape)

In [ ]:
# fill missing values in the covariates with the median
cols=["std_times_between_feedings_in_h", "BCQ_Attunement"]
data_rhythmicity[cols] = data_rhythmicity[cols].fillna(data_rhythmicity[cols].median())

In [ ]:
# linear mixed effects model for sleep rhythmicity and observed features rhythmicity
model1 = smf.mixedlm("CFI ~ R2_abundance + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity, 
                    groups=data_rhythmicity["infant_id"]).fit()
print(model1.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Shannon entropy rhythmicity
model2 = smf.mixedlm("CFI ~ R2_abundance_evenness + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity, 
                    groups=data_rhythmicity["infant_id"]).fit()
print(model2.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Pielou evenness rhythmicity
model3 = smf.mixedlm("CFI ~ R2_evenness + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity, 
                    groups=data_rhythmicity["infant_id"]).fit()
print(model3.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Faith phylogenetic diversity rhythmicity
model4 = smf.mixedlm("CFI ~ R2_biodiversity + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity, 
                    groups=data_rhythmicity["infant_id"]).fit()
print(model4.summary())

In [ ]:
# scatter plots of alpha diversity rhythmicity scores vs sleep rhythmicity
fig, axs = plt.subplots(1, 4, figsize=(16, 6))

scatterplot(data_rhythmicity, "R2_abundance", "CFI", "timepoint", 
                    "", "Sleep rhythmicity: circadian function index (CFI)", "Alpha diversity rhythmicity: \n cosine fit score of observed features",
                    timepoints_colors, axs[0], loc_position = 'upper right')

scatterplot(data_rhythmicity, "R2_abundance_evenness", "CFI", "timepoint", 
                    "", "", "Alpha diversity rhythmicity: \n cosine fit score of Shannon entropy",
                    timepoints_colors, axs[1], loc_position=None)

scatterplot(data_rhythmicity, "R2_evenness", "CFI", "timepoint", 
                    "", "", "Alpha diversity rhythmicity: \n cosine fit score of Pielou evenness",
                    timepoints_colors, axs[2], loc_position=None)

scatterplot(data_rhythmicity, "R2_biodiversity", "CFI", "timepoint", 
                    "", "", "Alpha diversity rhythmicity: \n cosine fit score of Faith's \n phylogenetic diversity",
                    timepoints_colors, axs[3], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/alpha_diversity_rhythmicity_sleep.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots
models = [model1, model2, model3, model4]
titles = ["Sleep rhythmicity: circadian function index (CFI)", "", "", ""]
plot_estimates_vertically(models, titles, f"{figures_path}/alpha_diversity_rhythmicity_sleep_estimates.pdf")

## Individual genera rhythmicity along age

In [ ]:
# filter data for genera rhythmicity values
R2_genus_cols = ['R2_veillo', 'R2_bifido', 'R2_esche', 'R2_bacteroides', 'R2_clostridium']
data_R2_genus = md_ages.dropna(subset=R2_genus_cols)

In [ ]:
# linear mixed effects model for Veillonella rhythmicity
model_ve = smf.mixedlm("R2_veillo ~ age_days + sex",
                    data_R2_genus, 
                    groups=data_R2_genus["infant_id"]).fit()
print(model_ve.summary())

In [ ]:
# linear mixed effects model for Bifidobacterium rhythmicity
model_bi = smf.mixedlm("R2_bifido ~ age_days + sex",
                    data_R2_genus, 
                    groups=data_R2_genus["infant_id"]).fit()
print(model_bi.summary())

In [ ]:
# linear mixed effects model for Escherichia-Shigella rhythmicity
model_es = smf.mixedlm("R2_esche ~ age_days + sex",
                    data_R2_genus, 
                    groups=data_R2_genus["infant_id"]).fit()
print(model_es.summary())

In [ ]:
# linear mixed effects model for Bacteroides rhythmicity
model_ba = smf.mixedlm("R2_bacteroides ~ age_days + sex",
                    data_R2_genus, 
                    groups=data_R2_genus["infant_id"]).fit()
print(model_ba.summary())

In [ ]:
# linear mixed effects model for Clostridium rhythmicity
model_cl = smf.mixedlm("R2_clostridium ~ age_days + sex",
                    data_R2_genus, 
                    groups=data_R2_genus["infant_id"]).fit()
print(model_cl.summary())

In [ ]:
# scatterplots of genera rhythmicity scores over age
fig, axs = plt.subplots(1, 5, figsize=(17, 5))

scatterplot(data_R2_genus, "age_days", "R2_veillo", "timepoint", 
                    "", "$\it{Veillonella}$ rhythmicity: \n cosine fit score of the $\it{Veillonella}$ genus", "Age (days)",
                    timepoints_colors, axs[0], loc_position='upper left')

scatterplot(data_R2_genus, "age_days", "R2_bifido", "timepoint", 
                    "", "$\it{Bifidobacterium}$  rhythmicity: \n cosine fit score of the $\it{Bifidobacterium}$ genus", "Age (days)",
                    timepoints_colors, axs[1], loc_position =None)

scatterplot(data_R2_genus, "age_days", "R2_esche", "timepoint", 
                    "", "$\it{Escherichia-Shigella}$ rhythmicity: cosine \n fit score of the $\it{Escherichia-Shigella}$ genus", "Age (days)",
                    timepoints_colors, axs[2], loc_position=None)

scatterplot(data_R2_genus, "age_days", "R2_bacteroides", "timepoint", 
                    "", "$\it{Bacteroides}$ rhythmicity: \n cosine fit score of the $\it{Bacteroides}$ genus", "Age (days)",
                    timepoints_colors, axs[3], loc_position=None)

scatterplot(data_R2_genus, "age_days", "R2_clostridium", "timepoint", 
                    "", "$\it{Clostridium}$ rhythmicity: \n cosine fit score of the $\it{Clostridium}$ genus", "Age (days)",
                    timepoints_colors, axs[4], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/genera_rhythmicity.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots
models = [model_ve, model_bi, model_es, model_ba, model_cl]
titles = ["$\it{Veillonella}$ rhythmicity: \n cosine fit of the \n $\it{Veillonella}$ genus",
          "$\it{Bifidobacterium}$ rhythmicity: \n cosine fit of the \n $\it{Bifidobacterium}$ genus",
          "$\it{Escherichia-Shigella}$ \n rhythmicity: cosine fit of the \n $\it{Escherichia-Shigella}$ genus",
          "$\it{Bacteroides}$ rhythmicity: \n cosine fit of the \n $\it{Bacteroides}$ genus",
          "$\it{Clostridium}$ rhythmicity: \n cosine fit of the \n $\it{Clostridium}$ genus"]
x_pos = [0.1, 0.1, 0.3, 0.01, 0.2]
plot_estimates(models, titles, x_pos, f"{figures_path}/genera_rhythmicity_estimates.pdf")

## Individual genera rhythmicity along sleep rhythmicity

In [ ]:
# filter data for the combined analysis
data_rhythmicity_genus = md_ages.dropna(subset=R2_genus_cols + ["CFI"]).copy()
print(data_rhythmicity_genus.shape)

In [ ]:
# fill missing values in the covariates with the median
data_rhythmicity_genus[cols] = data_rhythmicity_genus[cols].fillna(data_rhythmicity_genus[cols].median())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Bifidobacterium rhythmicity
model_bi_s = smf.mixedlm("CFI ~ R2_bifido + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity_genus, 
                    groups=data_rhythmicity_genus["infant_id"]).fit()
print(model_bi_s.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Veillonella rhythmicity
model_ve_s = smf.mixedlm("CFI ~ R2_veillo + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity_genus, 
                    groups=data_rhythmicity_genus["infant_id"]).fit()
print(model_ve_s.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Escherichia-Shigella rhythmicity
model_es_s = smf.mixedlm("CFI ~ R2_esche + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity_genus, 
                    groups=data_rhythmicity_genus["infant_id"]).fit()
print(model_es_s.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Bacteroides rhythmicity
model_ba_s = smf.mixedlm("CFI ~ R2_bacteroides + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity_genus, 
                    groups=data_rhythmicity_genus["infant_id"]).fit()
print(model_ba_s.summary())

In [ ]:
# linear mixed effects model for sleep rhythmicity and Clostridium rhythmicity
model_cl_s = smf.mixedlm("CFI ~ R2_clostridium + age_days + sex + std_times_between_feedings_in_h + BCQ_Attunement",
                    data_rhythmicity_genus, 
                    groups=data_rhythmicity_genus["infant_id"]).fit()
print(model_cl_s.summary())

In [ ]:
# scatterplots of genera rhythmicity scores vs sleep rhythmicity
fig, axs = plt.subplots(1, 5, figsize=(17, 5))

scatterplot(data_rhythmicity_genus, "R2_veillo", "CFI", "timepoint", 
                    "", "Sleep rhythmicity: \n circadian function index (CFI)", "$\it{Veillonella}$ rhythmicity: \n cosine fit score of the \n $\it{Veillonella}$ genus",
                    timepoints_colors, axs[0], loc_position='lower right')

scatterplot(data_rhythmicity_genus, "R2_bifido", "CFI", "timepoint", 
                    "", "", 
                    "$\it{Bifidobacterium}$ rhythmicity: \n cosine fit score of the \n $\it{Bifidobacterium}$ genus",
                    timepoints_colors, axs[1], loc_position=None)

scatterplot(data_rhythmicity_genus, "R2_esche", "CFI", "timepoint", 
                    "", "", "$\it{Escherichia-Shigella}$ rhythmicity: \n cosine fit score of the \n $\it{Escherichia-Shigella}$ genus",
                    timepoints_colors, axs[2], loc_position=None)

scatterplot(data_rhythmicity_genus, "R2_bacteroides", "CFI", "timepoint", 
                    "", "", "$\it{Bacteroides}$ rhythmicity: \n cosine fit score of the \n $\it{Bacteroides}$ genus",
                    timepoints_colors, axs[3], loc_position=None)

scatterplot(data_rhythmicity_genus, "R2_clostridium", "CFI", "timepoint", 
                    "", "", "$\it{Clostridium}$ rhythmicity: \n cosine fit score of the \n $\it{Clostridium}$ genus",
                    timepoints_colors, axs[4], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/genera_rhythmicity_sleep.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots
models = [model_ve_s, model_bi_s, model_es_s, model_ba_s, model_cl_s]
titles = ["Sleep rhythmicity: circadian function index (CFI)", "", "", "", ""]
plot_estimates_vertically(models, titles, f"{figures_path}/genera_rhythmicity_sleep_estimates.pdf")

## Fig. 2

In [ ]:
# Fig. 2a
fig, axs = plt.subplots(1, 2, figsize=(10, 7))

scatterplot(data_R2, "age_days", "R2_abundance", "timepoint", 
                    "", "Alpha diversity rhythmicity: \n cosine fit score of observed features", "Age (days)\n \n ",
                    timepoints_colors, axs[0], loc_position = 'upper center')

scatterplot(data_R2_genus, "age_days", "R2_bacteroides", "timepoint", 
                    "", "$\it{Bacteroides}$ rhythmicity: \n cosine fit score of the $\it{Bacteroides}$ genus", "Age (days)\n \n ",
                    timepoints_colors, axs[1], loc_position=None)

plt.subplots_adjust(wspace=0.4)

plt.savefig(f"{figures_path}/fig2a.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. 2b
fig, axs = plt.subplots(1, 3, figsize=(15, 7))

scatterplot(data_CFI, "age_days", "CFI", "timepoint", 
                    "", "Sleep rhythmicity:  circadian function index (CFI)", "Age (days)",
                    timepoints_colors, axs[0])

scatterplot(data_rhythmicity, "R2_abundance",  "CFI", "timepoint", 
                   "", "", "Alpha diversity rhythmicity: \n cosine fit score of observed features",
                   timepoints_colors, axs[1],loc_position = 'lower right')

scatterplot(data_rhythmicity_genus, "R2_bacteroides", "CFI", "timepoint", 
                    "", "", "$\it{Bacteroides}$ rhythmicity: cosine \n fit score of the $\it{Bacteroides}$ genus",
                    timepoints_colors, axs[2], loc_position= None)

plt.savefig(f"{figures_path}/fig2b.pdf", dpi=300, bbox_inches='tight')
plt.show()
